# 🦵 RSNA Knee Abnormality Detection: Rare-Finding Specialist Model

This notebook executes the **Fold 0 Smoke Check** for our Rare-Finding Specialist Model:
- **Hypothesis:** Minority abnormalities (`Fracture`, `MCL`, `Contusion`, `Lateral Meniscus`) are starved of gradients under standard BCE. By employing **Asymmetric Focal Loss (ASL)** with margin shifting and class-balancing on **RadImageNet ResNet50**, we boost minority-class sensitivity without degrading global consensus.
- **Architecture:** RadImageNet ResNet50 backbone + 6-slot anatomical `SlotHead` attention.
- **Validation:** 5-fold Multilabel Stratified Split grouped by report hash to prevent label leakage.


In [ ]:
import os, sys, time, gc, glob, hashlib, re, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import timm
import pydicom

print(f'PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


In [ ]:
# Competition Constants
TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 5
LR_HEAD = 1e-4
LR_BACKBONE = 2e-5

def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
seed_everything(SEED)


In [ ]:
# Locate Dataset Roots
def find_comp_root():
    for p in [Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
              Path('/kaggle/input/rsna-knee-abnormality-detection'), Path('input')]:
        if (p / 'train.csv').exists(): return p
    raise FileNotFoundError('Competition root not found')

COMP_ROOT = find_comp_root()
print('Competition root:', COMP_ROOT)

# Find RadImageNet Weights
def find_radimagenet_weights():
    candidates = [
        Path('/kaggle/input/datasets/marwanmath/resnet-50-radimagenet-marwan/ResNet50.pt'),
        Path('/kaggle/input/resnet-50-radimagenet-marwan/ResNet50.pt'),
    ]
    for c in candidates:
        if c.exists(): return c
    for c in glob.glob('/kaggle/input/**/ResNet50.pt', recursive=True):
        return Path(c)
    return None

RAD_WEIGHTS = find_radimagenet_weights()
print('RadImageNet weights:', RAD_WEIGHTS)


In [ ]:
# Load Training Data & Labels
train_csv = COMP_ROOT / 'train.csv'
df = pd.read_csv(train_csv)
print(f'Loaded {len(df)} studies from {train_csv}')

# Load precomputed high-quality LLM labels if available
llm_csv_candidates = [
    Path('/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv'),
    Path('/kaggle/input/rsna-knee-llm-labels/report_labels_v2.csv'),
]
llm_df = None
for c in llm_csv_candidates:
    if c.exists():
        llm_df = pd.read_csv(c)
        print(f'Loaded LLM report labels from {c}')
        break

if llm_df is not None:
    for col in TARGET_COLS:
        if col in df.columns:
            df[col] = df[col].combine_first(llm_df[col])
        else:
            df[col] = llm_df[col]
        if f'{col}__conf' in llm_df.columns:
            df[f'{col}__conf'] = llm_df[f'{col}__conf']
        else:
            df[f'{col}__conf'] = 1.0

# Generate Report-Grouped Fold Split
df['report_hash'] = df['Report'].fillna('').apply(lambda r: hashlib.md5(str(r).encode('utf-8')).hexdigest())
np.random.seed(SEED)
unique_hashes = df['report_hash'].unique()
np.random.shuffle(unique_hashes)
hash_to_fold = {h: i % 5 for i, h in enumerate(unique_hashes)}
df['fold'] = df['report_hash'].map(hash_to_fold)
print('Fold distribution:', df['fold'].value_counts().to_dict())


In [ ]:
# Asymmetric Focal Loss (ASL) with Margin Shift
class AsymmetricFocalLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, eps=1e-8, class_weights=None):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps
        self.class_weights = class_weights

    def forward(self, logits, targets, weights=None):
        p = torch.sigmoid(logits)
        xs_pos = p
        xs_neg = 1.0 - p
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)
        los_pos = targets * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1.0 - targets) * torch.log(xs_neg.clamp(min=self.eps))
        loss = los_pos + los_neg
        if self.gamma_neg > 0 or self.gamma_pos > 0:
            pt0 = xs_pos * targets
            pt1 = xs_neg * (1.0 - targets)
            pt = pt0 + pt1
            one_sided_gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
            one_sided_w = torch.pow(1.0 - pt, one_sided_gamma)
            loss = loss * one_sided_w
        if self.class_weights is not None:
            loss = loss * self.class_weights.unsqueeze(0)
        if weights is not None:
            loss = loss * weights
        return -loss.mean()

# Compute Inverse Frequency Weights
pos_counts = [(df[c] >= 0.5).sum() for c in TARGET_COLS]
inv_weights = torch.tensor([(len(df) / max(cnt, 1)) ** 0.35 for cnt in pos_counts], dtype=torch.float32)
inv_weights = (inv_weights / inv_weights.mean()).to(device)
for c, w, cnt in zip(TARGET_COLS, inv_weights.cpu().numpy(), pos_counts):
    print(f'{c:20s}: {cnt:4d} pos ({cnt/len(df)*100:5.2f}%) -> Weight: {w:.3f}')


In [ ]:
# Model Architecture: RadImageNet ResNet50 + Anatomical SlotHead
SLOT_PRIOR_TABLE = {
    'ACL': (0, 3, 5), 'MCL': (1, 4), 'Medial Meniscus': (0, 1, 3, 4),
    'Lateral Meniscus': (0, 1, 3, 4), 'Medial OA': (1, 4, 5),
    'Lateral OA': (1, 4, 5), 'PF OA': (0, 2, 5), 'Effusion': (0, 2),
    'Synovitis': (0, 2), "Baker's": (0,), 'Contusion': (0, 1, 2),
    'Fracture': (0, 1, 2, 4, 5),
}

class SlotHead(nn.Module):
    def __init__(self, dim=2048, n_slots=6, n_targets=12, hidden=256, dropout=0.2):
        super().__init__()
        self.hidden = hidden
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slots, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_targets, hidden) * 0.02)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hidden, n_targets)
        prior_m = torch.zeros(n_targets, n_slots)
        for i, tgt in enumerate(TARGET_COLS):
            for s_idx in SLOT_PRIOR_TABLE.get(tgt, ()):
                if s_idx < n_slots: prior_m[i, s_idx] = 0.55
        self.register_buffer('slot_prior', prior_m)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb.unsqueeze(0)
        att = torch.einsum('bsh,oh->bos', h, self.query) / (self.hidden**0.5) + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4)
        att = torch.softmax(att, dim=-1)
        ctx = self.drop(torch.einsum('bos,bsh->boh', att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(dim=-1) + self.out.bias

class SpecialistModel(nn.Module):
    def __init__(self, rad_weights=None):
        super().__init__()
        self.backbone = timm.create_model('resnet50', pretrained=(rad_weights is None), num_classes=0)
        if rad_weights is not None and Path(rad_weights).exists():
            st = torch.load(rad_weights, map_location='cpu')
            st = st.get('state_dict', st.get('model', st))
            st = {k.replace('module.', '').replace('backbone.', ''): v for k, v in st.items()}
            self.backbone.load_state_dict(st, strict=False)
            print('Loaded RadImageNet ResNet50 pretrained weights!')
        self.head = SlotHead(dim=self.backbone.num_features, n_slots=6, n_targets=12)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1))

    def forward(self, imgs, mask):
        b, s, c, h, w = imgs.shape
        x = (imgs - self.mean) / self.std
        feat = self.backbone(x.view(b * s, c, h, w)).view(b, s, -1)
        return self.head(feat, mask)


In [ ]:
# Summary Receipt
import json
print('\n--- Ready for Fold 0 Training Execution ---')
print('Target Minority Findings: Fracture, MCL, Contusion, Lateral Meniscus')
print('Loss: Asymmetric Focal Loss (gamma_neg=4.0, gamma_pos=1.0, margin=0.05)')
receipt = {'status': 'READY', 'fold': 0, 'loss': 'ASL', 'backbone': 'RadImageNet ResNet50'}
Path('/kaggle/working/specialist_receipt.json').write_text(json.dumps(receipt, indent=2))
print('Specialist setup complete. Wrote /kaggle/working/specialist_receipt.json')
